# Get started with the Marmoset Subcortical Cell Atlas data

This is a basic tutorial using the scanpy package to explore the marmoset subcortical cell atlas data. The annotation for the data is ongoing and therefore please load the newest annotation file every time your run this.

## import packages

In [ ]:
import scanpy as sc
import pandas as pd
from pathlib import Path
import os
from matplotlib.pyplot import rc_context
import matplotlib.pyplot as plt

sc.set_figure_params(frameon=False, dpi_save=400, transparent = False, fontsize=8)

## set up data paths

In [ ]:
pdir = '/jukebox/krienen/marm_hmba_integration/250602_reprocess_and_recluster'
anno_table_path = Path(pdir, 'metadata', 'subcortex_anno_table.csv')
data_path = Path(pdir,'data', 'rna', 'rna_clean_0715.h5ad')
output_dir = Path(pdir, 'analysis','work') ## please change this to /path/to/your/directory
os.chdir(output_dir)

## read in all data needed

In [ ]:
anno_table = pd.read_csv(anno_table_path, index_col=0)
adata = sc.read(data_path, backed = 'r') ## backed ='r' saves memory and time to load the data; only load data that you need to memory. 
cols = ['Subcortex_Class_v4', 'Subcortex_Group_v4', 'NT'] 
adata.obs = adata.obs.drop(columns = cols, errors='ignore')

## the july clean h5ad dropped some of the junk clusters so make sure that anno_table only contains clusters that are in adata
anno_table = anno_table.loc[adata.obs['Cluster_v4'].unique()]
## also make sure that we load in the data fully so that we can modify the obs dataframe
adata.obs = adata.obs.merge(anno_table[cols], left_on = 'Cluster_v4', right_index=True)

## metadata columns
 - Cluster_v4: Frozen marmoset subcortical cluster id. this will be the basis of all cell type annotation and will not change until we get more data.
 - Cluster_v3, cluster_id_v3: Frozen marmoset BG cluster id. The information in these two columns are the same. Cluster_v3 has "cluster_" and cluster_id_v3 has "marmoset-" in front of the cluster number. cluster_id_v3 is kept because the BG annotation table uses this convention to distinguish between species.
 - BG_Neighborhood, BG_Class, BG_Subclass, BG_Group: cells included in the BG preprint has these BG annotations harmonized across species
 - Region_v4, Subcortex_Class_v4, Subcortex_Group_v4, Subcortex_Subclass_hcpc_v4: current working labels updated from the anno_table above. Please join the latest subcortical annotation table and overwrite these columns if you would like to incorporate the newest cell types. 
	- Subcortex_Class_v4, Subcortex_Group_v4 are somewhat manually curated but still work in progress
	- Subcortex_Subclass_hcpc_v4: a computational hierarchical clustering approach using the HCPC package to group the clusters. This is not meant to be used for annotation and only included as a reference. 
 - Mapmycell results with various reference datasets
    - reference datasets: mouse abc atlas from Yao et al. 2023; human atlas from Siletti et al. 2023

## Visaulize the cell types on UMAP embeddings

In [ ]:
sc.pl.embedding(
            adata,
            basis='X_umap_clean',
            color=['Subcortex_Class_v4'],
            frameon=False,
            ncols=1,
            legend_fontoutline=2,
            legend_fontweight='bold',
        )

## Plot genes expression

In [ ]:
genes = [
    "SLC17A7",
    "SLC17A6",
    "GAD1",
    "GAD2",
    "MOG",
    "MOBP",
    "AQP4",
    "FOXP2",
]
with rc_context({"figure.figsize": (3, 3)}):
    ax = sc.pl.embedding(
        adata,
        basis='X_umap_donor',
        color=genes,
        frameon=False,
        ncols=4,
        vmax="p99",
        show=False
    )
    # plt.savefig("umap_gene_expression.png", dpi=400, bbox_inches='tight')
    # plt.close()

You will see that these expression plotted are raw counts. We can also store them as a layer and log normalize them. Note that this step might take a while to run given the current size of the data

In [ ]:
adata.layers['UMIs'] = adata.X.copy() ## store the raw data
sc.pp.normalize_total(adata, target_sum=1e6)
sc.pp.log1p(adata)

### Dotplot

Another way to visulize gene expression

In [ ]:
sc.pl.dotplot(adata, genes, "Subcortex_Class_v4", dendrogram=True)

You can refer to the scanpy tutorials [here](https://scanpy.readthedocs.io/en/stable/tutorials/plotting/core.html) for more plotting functionalities. 